In [1]:
# Cell 1 — Verify the DermNet dataset

from pathlib import Path

# Dataset location
DATA_DIR = Path.home() / "Desktop" / "dermai" / "data" / "dermnet"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

print("Dataset path:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

for split in ["train", "test"]:
    split_dir = DATA_DIR / split

    print(f"\n{'=' * 45}")
    print(f"{split.upper()} DATASET")
    print(f"{'=' * 45}")

    if not split_dir.exists():
        print(f"Folder not found: {split_dir}")
        continue

    total_images = 0
    class_count = 0

    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            count = sum(
                1 for p in class_dir.rglob("*")
                if p.is_file()
                and p.suffix.lower() in IMAGE_EXTENSIONS
            )

            print(f"{class_dir.name}: {count} images")
            total_images += count
            class_count += 1

    print("\nNumber of classes:", class_count)
    print("Total images:", total_images)

Dataset path: C:\Users\ASUS\Desktop\dermai\data\dermnet
Dataset exists: True

TRAIN DATASET
Acne and Rosacea Photos: 840 images
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions: 1149 images
Atopic Dermatitis Photos: 489 images
Bullous Disease Photos: 448 images
Cellulitis Impetigo and other Bacterial Infections: 288 images
Eczema Photos: 1235 images
Exanthems and Drug Eruptions: 404 images
Hair Loss Photos Alopecia and other Hair Diseases: 239 images
Herpes HPV and other STDs Photos: 405 images
Light Diseases and Disorders of Pigmentation: 568 images
Lupus and other Connective Tissue diseases: 420 images
Melanoma Skin Cancer Nevi and Moles: 463 images
Nail Fungus and other Nail Disease: 1040 images
Poison Ivy Photos and other Contact Dermatitis: 260 images
Psoriasis pictures Lichen Planus and related diseases: 1405 images
Scabies Lyme Disease and other Infestations and Bites: 431 images
Seborrheic Keratoses and other Benign Tumors: 1371 images
Systemic Disease: 606 im

In [2]:
# Cell 2 — Check PyTorch and GPU availability

import sys
import torch
import torchvision

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU memory: {gpu_memory:.2f} GB")

    DEVICE = torch.device("cuda")
else:
    print("GPU not detected. Training would use the CPU.")
    DEVICE = torch.device("cpu")

print("\nSelected device:", DEVICE)

Python version: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
PyTorch version: 2.6.0+cu124
Torchvision version: 0.21.0+cu124

CUDA available: True
GPU name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Number of GPUs: 1
GPU memory: 6.00 GB

Selected device: cuda


In [3]:
# Cell 3 — Load pretrained ConvNeXt-Tiny

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ConvNeXt_Tiny_Weights

# Configuration
NUM_CLASSES = 23
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ImageNet weights
weights = ConvNeXt_Tiny_Weights.DEFAULT
model = models.convnext_tiny(weights=weights)

# Replace the final classification layer
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

# Move model to GPU
model = model.to(DEVICE)

print("Model: ConvNeXt-Tiny")
print("Pretrained weights: ImageNet-1K")
print("Number of classes:", NUM_CLASSES)
print("Final layer:", model.classifier[2])
print("Device:", DEVICE)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Model loaded successfully!")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\ASUS/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [01:31<00:00, 1.25MB/s] 


Model: ConvNeXt-Tiny
Pretrained weights: ImageNet-1K
Number of classes: 23
Final layer: Linear(in_features=768, out_features=23, bias=True)
Device: cuda
Total parameters: 27,837,815
Model loaded successfully!


In [4]:
# Cell 4 — Load and preprocess the DermNet dataset

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Dataset paths
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# Preprocessing for pretrained ConvNeXt-Tiny
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = weights.transforms()

# Load datasets
train_dataset = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=str(TEST_DIR),
    transform=test_transform
)

# Verify class mapping
assert train_dataset.class_to_idx == test_dataset.class_to_idx, (
    "Train and test class mappings do not match!"
)

# DataLoaders
BATCH_SIZE = 8
NUM_WORKERS = 0  # Safe default for Windows/Jupyter

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Number of classes:", len(train_dataset.classes))
print("Classes:", train_dataset.classes)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

# Check one batch
images, labels = next(iter(train_loader))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image tensor device before GPU transfer:", images.device)

Training images: 15557
Testing images: 4002
Number of classes: 23
Classes: ['Acne and Rosacea Photos', 'Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions', 'Atopic Dermatitis Photos', 'Bullous Disease Photos', 'Cellulitis Impetigo and other Bacterial Infections', 'Eczema Photos', 'Exanthems and Drug Eruptions', 'Hair Loss Photos Alopecia and other Hair Diseases', 'Herpes HPV and other STDs Photos', 'Light Diseases and Disorders of Pigmentation', 'Lupus and other Connective Tissue diseases', 'Melanoma Skin Cancer Nevi and Moles', 'Nail Fungus and other Nail Disease', 'Poison Ivy Photos and other Contact Dermatitis', 'Psoriasis pictures Lichen Planus and related diseases', 'Scabies Lyme Disease and other Infestations and Bites', 'Seborrheic Keratoses and other Benign Tumors', 'Systemic Disease', 'Tinea Ringworm Candidiasis and other Fungal Infections', 'Urticaria Hives', 'Vascular Tumors', 'Vasculitis Photos', 'Warts Molluscum and other Viral Infections']
Training batche

In [6]:
# Cell 5 — Configure ConvNeXt-Tiny Training

import torch
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# 1. Freeze the pretrained feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# 2. Keep the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# 3. Loss function
criterion = nn.CrossEntropyLoss()

# 4. Optimizer
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

# 5. Training configuration
NUM_EPOCHS = 20

# 6. Learning-rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

# 7. Mixed precision for GPU memory efficiency
if DEVICE.type == "cuda":
    scaler = torch.amp.GradScaler("cuda")
else:
    scaler = None

# 8. Count parameters
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 45)
print("CONVNEXT-TINY TRAINING CONFIGURATION")
print("=" * 45)
print("Device:", DEVICE)
print("Number of classes:", NUM_CLASSES)
print("Total epochs:", NUM_EPOCHS)
print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print("Optimizer: AdamW")
print("Learning rate: 0.001")
print("Weight decay: 0.0001")
print("Loss function: CrossEntropyLoss")
print("Scheduler: CosineAnnealingLR")
print("Mixed precision:", DEVICE.type == "cuda")
print("\nTraining configuration ready!")

CONVNEXT-TINY TRAINING CONFIGURATION
Device: cuda
Number of classes: 23
Total epochs: 20
Trainable parameters: 19,223
Total parameters: 27,837,815
Optimizer: AdamW
Learning rate: 0.001
Weight decay: 0.0001
Loss function: CrossEntropyLoss
Scheduler: CosineAnnealingLR
Mixed precision: True

Training configuration ready!


## Cell 6 — Train ConvNeXt-Tiny for 20 epochs

In [7]:
# Cell 6A — Create a validation split from the training dataset

from torch.utils.data import random_split, DataLoader
import torch

VAL_RATIO = 0.15
BATCH_SIZE = 8

val_size = int(len(train_dataset) * VAL_RATIO)
train_size = len(train_dataset) - val_size

generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=generator
)

# Use a non-augmented transform for validation
val_dataset_full = datasets.ImageFolder(
    root=str(TRAIN_DIR),
    transform=test_transform
)

# Preserve the same indices but use validation preprocessing
val_subset = torch.utils.data.Subset(
    val_dataset_full,
    val_subset.indices
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images reserved for final evaluation:", len(test_dataset))

Training images: 13224
Validation images: 2333
Test images reserved for final evaluation: 4002


In [ ]:
# Cell 6B — Train ConvNeXt-Tiny for 20 epochs

import copy
import time
from pathlib import Path
import torch

# Checkpoint directory
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "convnext_tiny_dermnet_best.pth"

best_val_acc = 0.0
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # -------------------------
    # Training
    # -------------------------
    model.train()

    # Keep the frozen feature extractor in evaluation mode
    # so its BatchNorm/normalization behavior stays stable.
    model.features.eval()
    model.classifier.train()

    running_loss = 0.0
    running_correct = 0
    total_train = 0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train += batch_size

    train_loss = running_loss / total_train
    train_acc = running_correct / total_train

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    total_val = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            val_running_loss += loss.item() * batch_size
            val_running_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total_val += batch_size

    val_loss = val_running_loss / total_val
    val_acc = val_running_correct / total_val

    # Step learning-rate scheduler once per epoch
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    # Save the best model checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "class_to_idx": train_dataset.class_to_idx
        }, BEST_MODEL_PATH)

        saved_status = " | Best model saved"
    else:
        saved_status = ""

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc * 100:.2f}%"
        f"{saved_status}"
    )

elapsed = (time.time() - start_time) / 60

print("\nTraining completed.")
print(f"Best validation accuracy: {best_val_acc * 100:.2f}%")
print(f"Best checkpoint: {BEST_MODEL_PATH.resolve()}")
print(f"Total training time: {elapsed:.1f} minutes")

Epoch [01/20] Train Loss: 2.2597 | Train Acc: 33.70% | Val Loss: 2.1423 | Val Acc: 37.29% | Best model saved
Epoch [02/20] Train Loss: 1.9395 | Train Acc: 42.42% | Val Loss: 2.0426 | Val Acc: 39.56% | Best model saved
Epoch [03/20] Train Loss: 1.8333 | Train Acc: 44.84% | Val Loss: 2.0034 | Val Acc: 39.48%
Epoch [04/20] Train Loss: 1.7500 | Train Acc: 47.61% | Val Loss: 1.9921 | Val Acc: 40.85% | Best model saved
Epoch [05/20] Train Loss: 1.6903 | Train Acc: 48.88% | Val Loss: 1.9813 | Val Acc: 40.76%
Epoch [06/20] Train Loss: 1.6481 | Train Acc: 49.97% | Val Loss: 1.9823 | Val Acc: 41.11% | Best model saved
Epoch [07/20] Train Loss: 1.6036 | Train Acc: 51.01% | Val Loss: 2.0169 | Val Acc: 41.02%
Epoch [08/20] Train Loss: 1.5758 | Train Acc: 52.28% | Val Loss: 2.0010 | Val Acc: 40.63%
Epoch [09/20] Train Loss: 1.5412 | Train Acc: 52.66% | Val Loss: 1.9538 | Val Acc: 41.88% | Best model saved
Epoch [10/20] Train Loss: 1.5115 | Train Acc: 53.94% | Val Loss: 1.9660 | Val Acc: 42.31% | Bes

In [10]:
# Diagnostic — Check GPU and training progress

import torch
import os

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )
    print(
        "GPU memory reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )

print("\nHistory available:", "history" in globals())

if "history" in globals():
    print("Completed training epochs:", len(history["train_loss"]))
    print("Recorded validation epochs:", len(history["val_acc"]))

print("Best checkpoint exists:", BEST_MODEL_PATH.exists())

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory allocated: 0.12 GB
GPU memory reserved: 0.23 GB

History available: True
Completed training epochs: 20
Recorded validation epochs: 20
Best checkpoint exists: True


In [11]:
# Cell 7 — Verify all 20 recorded epochs

print("Configured epochs:", NUM_EPOCHS)
print("Training epochs recorded:", len(history["train_loss"]))
print("Validation epochs recorded:", len(history["val_acc"]))

print("\nLast 7 recorded epochs:")

for i in range(max(0, len(history["train_loss"]) - 7),
               len(history["train_loss"])):
    print(
        f"Epoch [{i + 1:02d}/20] | "
        f"Train Acc: {history['train_acc'][i] * 100:.2f}% | "
        f"Val Acc: {history['val_acc'][i] * 100:.2f}% | "
        f"Train Loss: {history['train_loss'][i]:.4f} | "
        f"Val Loss: {history['val_loss'][i]:.4f}"
    )

Configured epochs: 20
Training epochs recorded: 20
Validation epochs recorded: 20

Last 7 recorded epochs:
Epoch [14/20] | Train Acc: 56.61% | Val Acc: 42.31% | Train Loss: 1.4211 | Val Loss: 1.9614
Epoch [15/20] | Train Acc: 57.90% | Val Acc: 42.91% | Train Loss: 1.3938 | Val Loss: 1.9381
Epoch [16/20] | Train Acc: 57.60% | Val Acc: 41.96% | Train Loss: 1.3904 | Val Loss: 1.9457
Epoch [17/20] | Train Acc: 58.10% | Val Acc: 42.61% | Train Loss: 1.3775 | Val Loss: 1.9419
Epoch [18/20] | Train Acc: 58.40% | Val Acc: 42.56% | Train Loss: 1.3672 | Val Loss: 1.9416
Epoch [19/20] | Train Acc: 58.20% | Val Acc: 42.31% | Train Loss: 1.3641 | Val Loss: 1.9392
Epoch [20/20] | Train Acc: 58.82% | Val Acc: 42.31% | Train Loss: 1.3504 | Val Loss: 1.9397


In [12]:
# Cell 9 — Prepare ConvNeXt-Tiny for fine-tuning

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Load the best model from Phase 1
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print(
    "Checkpoint validation accuracy:",
    f'{checkpoint["val_accuracy"] * 100:.2f}%'
)

# Freeze all feature layers first
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the later feature stages
# ConvNeXt-Tiny features: stem, stage 1, stage 2, stage 3, stage 4
for param in model.features[5].parameters():
    param.requires_grad = True

for param in model.features[7].parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Reinitialize optimizer for fine-tuning
optimizer = AdamW(
    [
        {
            "params": model.features[5].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.features[7].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

# Fine-tuning settings
FINE_TUNE_EPOCHS = 15

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Mixed precision
scaler = torch.amp.GradScaler("cuda")

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nFine-tuning configuration ready")
print("Trainable parameters:", f"{trainable_params:,}")
print("Fine-tuning epochs:", FINE_TUNE_EPOCHS)
print("Backbone learning rate: 1e-5")
print("Classifier learning rate: 1e-4")
print("Device:", DEVICE)

Loaded checkpoint from epoch: 15
Checkpoint validation accuracy: 42.91%

Fine-tuning configuration ready
Trainable parameters: 25,125,911
Fine-tuning epochs: 15
Backbone learning rate: 1e-5
Classifier learning rate: 1e-4
Device: cuda


In [14]:
# Cell 10 — Fine-tune ConvNeXt-Tiny

import time
import copy
import torch
from pathlib import Path

FINE_TUNE_MODEL_PATH = MODEL_DIR / "convnext_tiny_dermnet_finetuned_best.pth"

fine_tune_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_finetune_val_acc = checkpoint["val_accuracy"]
patience = 5
epochs_without_improvement = 0

start_time = time.time()

for epoch in range(FINE_TUNE_EPOCHS):
    # -------------------------
    # Training
    # -------------------------
    model.train()

    # Keep frozen stages in evaluation mode
    for module in model.features:
        if not any(p.requires_grad for p in module.parameters()):
            module.eval()

    running_loss = 0.0
    running_correct = 0
    total_train = 0

    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total_train += batch_size

    train_loss = running_loss / total_train
    train_acc = running_correct / total_train

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    total_val = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            val_running_loss += loss.item() * batch_size
            val_running_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            total_val += batch_size

    val_loss = val_running_loss / total_val
    val_acc = val_running_correct / total_val

    scheduler.step(val_acc)

    fine_tune_history["train_loss"].append(train_loss)
    fine_tune_history["train_acc"].append(train_acc)
    fine_tune_history["val_loss"].append(val_loss)
    fine_tune_history["val_acc"].append(val_acc)

    improved = val_acc > best_finetune_val_acc

    if improved:
        best_finetune_val_acc = val_acc
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "val_accuracy": val_acc,
            "class_to_idx": train_dataset.class_to_idx
        }, FINE_TUNE_MODEL_PATH)

        status = " | Best fine-tuned model saved"
    else:
        epochs_without_improvement += 1
        status = ""

    print(
        f"Fine-tune Epoch [{epoch + 1:02d}/{FINE_TUNE_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc * 100:.2f}%"
        f"{status}"
    )

    if epochs_without_improvement >= patience:
        print("Early stopping: validation accuracy did not improve.")
        break

elapsed = (time.time() - start_time) / 60

print("\nFine-tuning finished.")
print(f"Best validation accuracy: {best_finetune_val_acc * 100:.2f}%")
print(f"Training time: {elapsed:.1f} minutes")
print("Best checkpoint:", FINE_TUNE_MODEL_PATH.resolve())

Fine-tune Epoch [01/15] Train Loss: 0.1990 | Train Acc: 94.02% | Val Loss: 1.8377 | Val Acc: 59.32% | Best fine-tuned model saved


KeyboardInterrupt: 